<a href="https://colab.research.google.com/github/Git-Hub-Ran/rag-qa-langchain/blob/Dev/rag_qa_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [72]:
!pip install "langchain==0.3.27" "langchain-community>=0.3.0" "langchain-core>=0.3.0" "langchain-openai>=0.3.0" langchain-text-splitters chromadb unstructured python-dotenv

In [73]:
#Connect to Google Drive to use API key:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [74]:
#Loading API key from .env:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/Colab Notebooks/openai-chatbot/.env")

True

In [84]:
from langchain_community.document_loaders import UnstructuredURLLoader

# Webites that will be saved as documents:
urls = [
    "https://www.governor.ny.gov/news", #New York state news website
    "https://www.gov.uk/government/news/government-launches-good-food-cycle-to-transform-britains-food-system" #Federal Government Press Release
]

#load documents:
loader = UnstructuredURLLoader(urls=urls)
docs = loader.load()

print(f"Number of documents that were loaded: {len(docs)}")

Number of documents that were loaded: 2


In [85]:
docs[0].page_content[:5000]  # Validation that the document was loaded : see the content of the document

'Pressroom\n\nOfficial news from the Office of the Governor\n\nMarch 02, 2026\n\nGovernor Hochul Highlights Chaos Created by Trump\'s Tariffs\n\nMeeting with business leaders, Governor Hochul emphasized the challenges and financial hardships small business owners faced over the past year from Trump’s illegal tariffs.\n\nPress Release\n\nFeatured News\n\nProtecting Basic Liberties in New York\n\nProtecting Basic Liberties in New York\n\nMarch 2, 2026 | 2:02 PM EST\n\nGovernor Hochul highlighted Border Patrol’s mismanaged release of Nurul Amin Shah Alam, a refugee from Myanmar who was partially blind and never returned to his family, and ICE’s violation of constitutional rights in their recent detainment of Columbia student Ellie Aghayeva.\n\nKeeping New Yorkers Safe as Iran Attacks Unfold\n\nKeeping New Yorkers Safe as Iran Attacks Unfold\n\nMarch 2, 2026 | 1:55 PM EST\n\nGovernor Hochul delivered remarks on the ongoing attacks in Iran.\n\nAnnual Chinatown Lunar New Year Parade\n\nAnnua

In [86]:
#Break the docs into smaller chunks:
from langchain_text_splitters import RecursiveCharacterTextSplitter #This text splitter is the recommended one for generic text.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,#The maximum size of a chunk
    chunk_overlap=100 #Target overlap between chunks. Overlapping chunks helps to mitigate loss of information when context is divided between chunks.
    )
chunks = text_splitter.split_documents(docs)

print(f"Number of chunks: {len(chunks)}")

Number of chunks: 22


In [87]:
#create embedding:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

#creating vector store with Chroma:
from langchain_community.vectorstores import Chroma

vector_store = Chroma.from_documents(
    documents=chunks, #every chunk in the list
    embedding=embeddings #the function that convert every chunk into a numeric vector
)
#Now the vector store is ready for semantic search

In [88]:
#Create a retriever from the vector store. The retriever can get a question and return an answer from the relevent chunks of documents:
retriever = vector_store.as_retriever( search_kwargs={"k": 3})

In [89]:
from langchain_openai import ChatOpenAI

#Creating LLM:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

In [90]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


#The question I want to ask :
# queries = [
#     "What are the main news topics on the New York governor website?",
#     "Who invented the chocolate chip cookie and where?"
# ]

queries_governor = [
    "What issue did Governor Hochul highlight regarding Trump's tariffs?",
    "What did Governor Hochul say about Border Patrol?",
    "What is the capital of France?" #This info doesn't exists in this website , so he should answer he don't know
]

queries_food_press = [
    "What is the main goal of the 'Good Food Cycle' initiative?",
    "Which outcomes does the Good Food Cycle framework identify?",
    "Why does the government consider food security important according to the press release?",
    "What is the capital of France?" #This info doesn't exists in this website , so he should answer he don't know
]

#define the prompt:
prompt = ChatPromptTemplate.from_messages([
   ("system",
    "You are a helpful assistant.\n"
    "Answer the question ONLY using the provided context.\n"
    "If the answer is not in the context, say 'I don’t know.'\n\n"
    "Context:\n{context}"
   ),
   ("human", "{input}")
])

# Create the answer-generation chain (LLM + prompt that processes retrieved documents):
question_answer_chain = create_stuff_documents_chain(llm, prompt)

# Build complete retrieval augmented generation (RAG) pipeline:
qa_chain = create_retrieval_chain(retriever, question_answer_chain)

In [91]:
#get answers on the first website:
for query in queries_governor:
    result = qa_chain.invoke({"input": query})

    print(f"\nQ: {query}")
    print(f"A: {result['answer']}\n")

    print("Retrieved Chunks:")
    for i, doc in enumerate(result["context"]):
        print(f"\nChunk {i+1}:")
        print(doc.page_content[:500])  # only part of the texts
        print("-" * 40)


Q: What issue did Governor Hochul highlight regarding Trump's tariffs?
A: Governor Hochul highlighted the challenges and financial hardships small business owners faced over the past year from Trump’s illegal tariffs.

Retrieved Chunks:

Chunk 1:
Pressroom

Official news from the Office of the Governor

March 02, 2026

Governor Hochul Highlights Chaos Created by Trump's Tariffs

Meeting with business leaders, Governor Hochul emphasized the challenges and financial hardships small business owners faced over the past year from Trump’s illegal tariffs.

Press Release

Featured News

Protecting Basic Liberties in New York

Protecting Basic Liberties in New York

March 2, 2026 | 2:02 PM EST

Governor Hochul highlighted Border Patrol’s mismana
----------------------------------------

Chunk 2:
Pressroom

Official news from the Office of the Governor

March 02, 2026

Governor Hochul Highlights Chaos Created by Trump's Tariffs

Meeting with business leaders, Governor Hochul emphasized the cha

In [83]:
#get an answers on the first website:
for query in queries_cookie:
    result = qa_chain.invoke({"input": query})

    print(f"\nQ: {query}")
    print(f"A: {result['answer']}\n")

    print("Retrieved Chunks:")
    for i, doc in enumerate(result["context"]):
        print(f"\nChunk {i+1}:")
        print(doc.page_content[:500])  # only part of the texts
        print("-" * 40)


Q: What is the main goal of the 'Good Food Cycle' initiative?
A: The main goal of the 'Good Food Cycle' initiative is to drive a generational change in the nation’s relationship with food by creating a thriving food sector that addresses challenges such as rising obesity rates and climate change impacts on production, while ensuring access to safe, affordable, healthy, convenient, and appealing food options for all.

Retrieved Chunks:

Chunk 1:
Press release

Government launches "Good Food Cycle" to transform Britain's food system

New "Good Food Cycle" framework serves up healthier eating, stronger food security and greener supply chains

The government has served up its new “Good Food Cycle” today (15 July) – a recipe aimed at driving a generational change in the nation’s relationship with food.

The Good Food Cycle identifies ten priority outcomes needed to build a thriving food sector while tackling challenges from rising obesity r
----------------------------------------

Chunk 2

We can see that the model answers questions using only the provided documents. It gives correct answers when information is in the text and says "I don’t know" when it is not. Chunks show where the information came from.